In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import re
from dotenv import load_dotenv

import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import time
from datetime import timedelta
from tqdm import tqdm

import yfinance as yf
from gnews import GNews

import warnings
warnings.filterwarnings('ignore')

## Parameters

In [2]:
load_dotenv()
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# if OPENAI_API_KEY is None:
#     raise ValueError("OPENAI_API_KEY is not found. Please check your .env file.")
# else:
#     print(f"OPENAI_API_KEY = {OPENAI_API_KEY[:8]}***** (Loaded Successfully)\n")

variables = []
with open("00 Variables.txt", "r") as f:
    lines = f.readlines()
    hashes = 1
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            hashes = len(s) - len(s.lstrip("#"))
            comment_indent = (hashes - 1) * 2
            print(" " * comment_indent + "\033[92m" + s + "\033[0m")
        elif "=" in s:
            var = s.split("=", 1)[0].strip()
            variables.append(var)
            try:
                exec(s)
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                value_part = s.split("#", 1)[0].strip()
                comment_part = s[s.find("#"):] if "#" in s else ""
                eval_value = f"{var} = {repr(eval(var))}"
                if comment_part:
                    print(" " * black_indent + eval_value + " \033[38;5;250m" + "  " + comment_part + "\033[0m")
                else:
                    print(" " * black_indent + eval_value)
            except Exception as e:
                black_indent = {1: 2, 2: 5, 3: 9, 4: 14}.get(hashes, (hashes - 1) * 2 + sum(range(hashes + 1)))
                print(" " * black_indent + s)

# Global Settings
  TICKER = '^GSPC'
  START_DATE = '2024-09-01'
  END_DATE = '2026-04-30'
  WINDOW_SIZE = 126
  REGIME_WINDOW = 126   # Lookback for regime stats (vol, quantile bands). Keep = WINDOW_SIZE so bands match the label
  HORIZON_DAYS = 1
  CRASH_PERCENTILE = 0.05
  MAX_WINDOWS = 10000
  DATE_INFO = True
  INDICATOR_SIZE = 'small'   # "small", "large"
  CHART_NUM = 4   # 9 panels split as [3, 2, 2, 2]
  TRANSPARENT = False
  CHART_DPI = 72   # 144 = original; 72 halves each edge and quarters figure memory
  NEWS_QUERY = 'Standard and Poors 500 stock index'
  MODEL_NAME = 'gpt-5-mini'   # "gpt-5-nano-2025-08-07", "gpt-5-mini-2025-08-07", "gpt-5-2025-08-07"
  EMBEDDING_MODEL = 'text-embedding-3-large'
  RUN_TIMESTAMP = '20260603 1232'   # This will be changed automatically
  N_PREV = 6   # Number of previous windows to provide context foBr
  USE_IMAGES = True   # If True, send images (multimodal model), else just text
# Path
  ## Layer 1
     L1_RESULTS = 'Results'
     L1_COMB

## Definition

In [3]:
def download_and_prepare_prices(ticker, start, end):
    """
    Downloads historical stock prices for a given ticker and prepares the DataFrame.
    Args:
        - ticker: str, stock ticker symbol (e.g., 'AAPL')
        - start: str, start date in 'YYYY-MM-DD' format
        - end: str, end date in 'YYYY-MM-DD' format
    Returns:
        - df: pd.DataFrame, DataFrame with columns ['Close', 'High', 'Low', 'Open', 'Volume']
    """

    df_raw = yf.download(ticker, start=start, end=end)

    if isinstance(df_raw.columns, pd.MultiIndex):
        df_raw.columns = [c[0] for c in df_raw.columns]
        print("Flattened MultiIndex columns:", df_raw.columns.tolist())
    df = df_raw.copy()

    expected_cols = ['Close', 'High', 'Low', 'Open', 'Volume']
    missing = [col for col in expected_cols if col not in df.columns]
    if missing:
        print("Columns in data:", df.columns.tolist())
        raise ValueError(f"Missing columns in data: {missing}")

    df = df[expected_cols]

    for col in expected_cols:
        if isinstance(df[col].dtype, pd.StringDtype):
            print(f"Column {col} is a DataFrame, converting to Series...")
            df[col] = df[col].iloc[:, 0]
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    return df

In [ ]:
def calculate_core_crash_indicators(df: pd.DataFrame):
    r = df.copy()
    r.columns = [str(col).lower() for col in r.columns]
    req = ['close', 'high', 'low', 'open', 'volume']
    for c in req:
        if c not in r.columns:
            raise ValueError(f"Missing required column: {c}")
        r[c] = pd.to_numeric(r[c], errors='coerce')
    if not isinstance(r.index, pd.DatetimeIndex):
        r.index = pd.to_datetime(r.index)
    r = r.sort_index()

    c, h, l, o, v = r['close'], r['high'], r['low'], r['open'], r['volume']
    eps = 1e-10
    n = REGIME_WINDOW

    def wilder(s, p):
        return s.ewm(
            alpha=1.0 / p,
            adjust=False
        ).mean()

    sma20 = c.rolling(20).mean()
    sma50 = c.rolling(50).mean()
    sma200 = c.rolling(200).mean()
    r['sma20'] = sma20
    r['sma50'] = sma50
    r['sma200'] = sma200

    std20 = c.rolling(20).std()
    r['bb_upper'] = sma20 + 2 * std20
    r['bb_lower'] = sma20 - 2 * std20
    r['bb_width'] = (r['bb_upper'] - r['bb_lower']) / (sma20 + eps)

    r['death_cross'] = ((sma50.shift(1) > sma200.shift(1)) & (sma50 < sma200)).astype(int)
    r['golden_cross'] = ((sma50.shift(1) < sma200.shift(1)) & (sma50 > sma200)).astype(int)

    r['logreturn'] = np.log(c / c.shift(1))
    logret = r['logreturn']
    r['hv_126d_ann'] = logret.rolling(n).std() * np.sqrt(252)
    r['hv_10d_ann'] = logret.rolling(10).std() * np.sqrt(252)
    r['hv_60d_ann'] = logret.rolling(60).std() * np.sqrt(252)

    log_hl = np.log(h / l)
    log_co = np.log(c / o)

    r['hv_parkinson_126d_ann'] = np.sqrt(
        (1.0 / (4.0 * np.log(2))) * (log_hl ** 2).rolling(n).mean()
    ) * np.sqrt(252)

    r['vol_gk'] = 0.5 * (log_hl ** 2).rolling(n).mean() - (2 * np.log(2) - 1) * (log_co ** 2).rolling(n).mean()
    r['hv_gk_126d_ann'] = r['vol_gk'].apply(lambda x: np.sqrt(x) if x > 0 else np.nan) * np.sqrt(252)


    log_ho = np.log(h / o)
    log_lo = np.log(l / o)
    r['vol_rs'] = (log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)).rolling(n).mean()
    r['hv_rs_126d_ann'] = r['vol_rs'].apply(lambda x: np.sqrt(x) if x > 0 else np.nan) * np.sqrt(252)

    ohlc_vols = ['hv_126d_ann', 'hv_parkinson_126d_ann', 'hv_gk_126d_ann', 'hv_rs_126d_ann']
    r['composite_vol'] = r[ohlc_vols].mean(axis=1)
    r['hv_composite_21d_ann'] = r['composite_vol'].rolling(21).std()

    historical_returns = logret
    historical_volatility = r['hv_126d_ann']
    for q in [0.01, 0.05, 0.95, 0.99]:
        pct = int(q * 100)
        r[f'log_return_1d_q{pct:02d}_roll126d'] = historical_returns.rolling(n).quantile(q)
        r[f'hv_126d_ann_q{pct:02d}_roll126d'] = historical_volatility.rolling(n).quantile(q)

    ema12 = c.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = c.ewm(span=26, adjust=False, min_periods=26).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    macd_hist = macd - macd_signal

    r['macd_12_26_norm'] = macd / (c + eps)
    r['macd_signal_9_norm'] = macd_signal / (c + eps)
    r['macd_hist_12_26_9_norm'] = macd_hist / (c + eps)

    r['roc_5d'] = ((c / c.shift(5)) - 1) * 100
    r['roc_10d'] = ((c / c.shift(10)) - 1) * 100
    r['roc_20d'] = ((c / c.shift(20)) - 1) * 100

    atr_period = 14
    previous_close = c.shift(1)
    tr = pd.concat([h - l, (h - previous_close).abs(), (l - previous_close).abs()], axis=1).max(axis=1)

    atr = wilder(tr, atr_period)
    r['tr_1d_points'] = tr
    r['atr_14d_points'] = atr

    r['tr_1d_norm'] = tr / (c + eps)
    r['atr_14d_norm'] = atr / (c + eps)

    r['tr_to_atr_14d'] = tr / atr

    mfm = ((c - l) - (h - c)) / (h - l + eps)
    cmf_num = (mfm * v).rolling(20).sum()
    cmf_den = v.rolling(20).sum().replace(0, np.nan)
    r['cmf_20d'] = cmf_num / cmf_den

    volume_mean_20d = (v.rolling(20).mean())
    volume_std_20d = (v.rolling(20).std())

    r['volume_z_20d'] = (v - volume_mean_20d) / (volume_std_20d + eps)

    signed_volume = np.sign(c.diff()) * v
    rolling_signed_volume = signed_volume.rolling(20).sum()
    rolling_total_volume = v.rolling(20).sum()
    r['obv_flow_20d_norm'] = rolling_signed_volume / rolling_total_volume .replace(0, np.nan)

    lowest_low = l.rolling(14).min()
    highest_high = h.rolling(14).max()
    r['stoch_k'] = 100 * (c - lowest_low) / (highest_high - lowest_low + eps)
    r['stoch_d'] = r['stoch_k'].rolling(3).mean()

    r['price_zscore_20d'] = (c - sma20) / (c.rolling(20).std() + eps)

    delta = c.diff()
    avg_gain = wilder(delta.clip(lower=0), 14)
    avg_loss = wilder((-delta).clip(lower=0), 14)
    r['rsi_14d'] = 100 - (100 / (1 + avg_gain / (avg_loss + eps)))

    tp = (h + l + c) / 3
    ma_tp = tp.rolling(20).mean()
    md = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    r['cci_20d'] = (tp - ma_tp) / (0.015 * md + eps)

    r['sharpe_20_ann_rf0'] = logret.rolling(20).mean() / (logret.rolling(20).std() + eps) * np.sqrt(252)

    r['dd_20'] = (c / c.rolling(20).max() - 1) * 100
    max_close_14 = c.rolling(14).max()
    drawdown = ((c - max_close_14) / max_close_14) * 100
    r['ulcer_index'] = np.sqrt((drawdown ** 2).rolling(14).mean())

    # === Tail risk ===
    # The deeper the tail, the longer the window it needs. The 20-day VaR/ES columns were
    # deleted: at n=20 the 5% "quantile" is just the 2nd smallest return, the 1% one is a
    # blend of the two smallest, and ceil(0.05*20)=1 made "ES" the rolling minimum.
    # The 126d levels are r_1pct / r_5pct above.
    # r['var_5pct_60'] = logret.rolling(60).quantile(0.05)

    def es(x, p=0.05):
        x = x[np.isfinite(x)]

        if len(x) == 0:
            return np.nan

        k = max(int(np.ceil(p * len(x))), 1)
        return np.sort(x)[:k].mean()

    r['es_5pct_126'] = logret.rolling(n).apply(lambda x: es(x, 0.05),raw=True )

    up = h.diff()
    down = -l.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=r.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=r.index)
    assert not ((plus_dm > 0) & (minus_dm > 0)).any(), "+DM and -DM both fired on one bar"

    plus_di = 100 * wilder(plus_dm, 14) / (r['atr_14d_points'] + eps)
    minus_di = 100 * wilder(minus_dm, 14) / (r['atr_14d_points'] + eps)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + eps)
    r['plus_di'] = plus_di
    r['minus_di'] = minus_di
    r['adx'] = wilder(dx, 14)

    r['skewness_126'] = logret.rolling(126).skew()
    r['kurtosis_126'] = logret.rolling(126).kurt()

    return r

In [5]:
def compute_crash_labels_and_stats(price_df, prediction_dates, window_size=WINDOW_SIZE, horizon_days=HORIZON_DAYS, crash_percentile=CRASH_PERCENTILE, close_col='close'):
    close = price_df[close_col]
    logret = np.log(close).diff()
    quantile_series = (
        logret.rolling(window_size, min_periods=window_size)
        .quantile(crash_percentile)
        .shift(1)
    )
    labels, min_future_logrets, thresholds = [], [], []
    for d in tqdm(prediction_dates, desc="Crash label stats"):
        if d not in price_df.index or pd.isna(quantile_series.loc[d]):
            labels.append(np.nan)
            min_future_logrets.append(np.nan)
            thresholds.append(np.nan)
            continue
        idx = price_df.index.get_loc(d)
        crash_thr = quantile_series.iloc[idx]
        future_idx = np.arange(idx+1, min(idx+1+horizon_days, len(close)))
        if len(future_idx) == 0:
            labels.append(np.nan)
            min_future_logrets.append(np.nan)
            thresholds.append(crash_thr)
            continue
        rets = logret.iloc[future_idx]
        min_ret = rets.min()
        crash = int(min_ret <= crash_thr)
        labels.append(crash)
        min_future_logrets.append(min_ret)
        thresholds.append(crash_thr)
    return labels, min_future_logrets, thresholds


In [6]:
def fetch_and_cache_news(query, start_date, end_date, cache_path: str, max_results=5, chunk_size=3):

    def remove_dates(text):
        date_patterns = [
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},\s+\d{4}',    # August 13, 2024 / Aug. 16, 2024
            r'\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}',     # 22-06-2012 or 22/06/2012
            r'\b\d{4}[-./]\d{1,2}[-./]\d{1,2}',     # # 2025.08.22 or 2025-08-22
            r'\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}',     # 22 August 2024
            r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?\s+\d{4}'     # August 2024
        ]
        combined_pattern = re.compile('|'.join(date_patterns), flags=re.IGNORECASE)
        if type(text)==str:
            value = combined_pattern.sub('', text)
        else:
            value = np.nan
        return value
    
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    gnews = GNews(language='en', country='US', max_results=max_results)
    all_news_list = []
    date_ranges = pd.date_range(start_date, end_date, freq=f'{chunk_size}D')
    for chunk_start in tqdm(date_ranges, desc=f"Fetching {chunk_size}-day chunks"):
        chunk_end = min(chunk_start + pd.Timedelta(days=chunk_size-1), pd.Timestamp(end_date))
        after_str = chunk_start.strftime('%Y-%m-%d')
        before_str = (chunk_end + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        try:
            articles = gnews.get_news(f"{query} after:{after_str} before:{before_str}")
            news_text = " ".join([a.get('title', '') for a in articles])[:500]
        except Exception as e:
            print(f"[WARN] Error fetching news for {after_str} - {chunk_end}: {e}")
            news_text = ""
        for single_day in pd.date_range(chunk_start, chunk_end, freq='D'):
            all_news_list.append({"Date": single_day, "News": news_text})
        time.sleep(1)
        news_df = pd.DataFrame(all_news_list).set_index('Date')
        news_df = news_df[~news_df.index.duplicated()]
        news_df['News'] = news_df['News'].apply(remove_dates)
        news_df.to_csv(cache_path)
    return news_df

## Data Collection

In [7]:
ANALYSIS_START_DATE = pd.Timestamp(START_DATE)
ANALYSIS_END_DATE = pd.Timestamp(END_DATE)

TECHNICAL_WARMUP_TRADING_DAYS = max(200, 2 * WINDOW_SIZE, 60)
QUANTILE_WARMUP_TRADING_DAYS = WINDOW_SIZE + 1
TOTAL_WARMUP_TRADING_DAYS = TECHNICAL_WARMUP_TRADING_DAYS + QUANTILE_WARMUP_TRADING_DAYS + 30

RAW_START_DATE = (ANALYSIS_START_DATE - pd.offsets.BDay(TOTAL_WARMUP_TRADING_DAYS)).strftime("%Y-%m-%d")

print(f"\nDownloading price data for {TICKER} from {RAW_START_DATE} to {END_DATE} ...")
print(f"Analysis START_DATE remains {START_DATE}; raw download starts earlier for rolling warm-up.")
df = download_and_prepare_prices(TICKER, RAW_START_DATE, END_DATE)
print("Price data loaded:", df.shape)
display(df)


Analysis START_DATE remains 2024-09-01; raw download starts earlier for rolling warm-up.


[*********************100%***********************]  1 of 1 completed

Flattened MultiIndex columns: ['Close', 'High', 'Low', 'Open', 'Volume']
Price data loaded: (809, 5)


,Close,High,Low,Open,Volume
Date,,,,,
2023-02-07,4164.000000,4176.540039,4088.389893,4105.350098,4355860000
2023-02-08,4117.859863,4156.850098,4111.669922,4153.470215,4029820000
2023-02-09,4081.500000,4156.229980,4069.669922,4144.250000,4270200000
2023-02-10,4090.459961,4094.360107,4060.790039,4068.919922,3891520000
2023-02-13,4137.290039,4138.899902,4092.669922,4096.620117,3448620000
...,...,...,...,...,...
2026-04-23,7108.399902,7147.779785,7046.549805,7118.799805,5307260000
2026-04-24,7165.080078,7168.589844,7112.819824,7136.479980,4608830000
2026-04-27,7173.910156,7178.740234,7146.720215,7152.720215,4783750000


In [10]:
# The first bar has no previous close and therefore no log return, so it is dropped
# explicitly. This used to happen as a side effect of adding a duplicate `return`
# column (identical to the `logreturn` the indicator function computes) purely so that
# dropna() would remove that one row.
df = df.iloc[1:].copy()

df_ind = calculate_core_crash_indicators(df)
df_ind_orig = df_ind.copy()
df_ind = df_ind.dropna().copy()

# No column may be an exact copy of another: `return`/`logreturn`, `return_5pct`/
# `var_5pct_20` and `return_1pct`/`var_1pct_20` were all duplicate pairs that went
# unnoticed for exactly this lack of a check.
dupes = df_ind.columns[df_ind.T.duplicated()].tolist()
assert not dupes, f"Duplicate indicator columns: {dupes}"

price_df = df_ind.copy()
price_df.index = pd.to_datetime(price_df.index)
price_df.columns = [c.lower() for c in price_df.columns]

logret = np.log(price_df['close']).diff()
quantile_series = (
    logret.rolling(WINDOW_SIZE, min_periods=WINDOW_SIZE)
    .quantile(CRASH_PERCENTILE)
    .shift(1)
)
valid_dates = quantile_series.dropna().index

In [11]:
labels, min_future_logrets, thresholds = compute_crash_labels_and_stats(
    price_df, 
    price_df.index, 
    window_size=WINDOW_SIZE, 
    horizon_days=HORIZON_DAYS, 
    crash_percentile=CRASH_PERCENTILE,
    close_col='close'
)

df_labels = pd.DataFrame({
    "date": price_df.index,
    "crash_label": labels,
    "min_future_logret": min_future_logrets,
    "crash_threshold": thresholds
}).set_index('date')

Crash label stats: 100%|██████████| 556/556 [00:00<00:00, 27924.53it/s]


In [12]:
df_full = pd.DataFrame(index=df.index.union(df_ind.index).union(df_labels.index).union(quantile_series.index))
df_full = df_full.assign(**df.to_dict('series'))
df_full = df_full.assign(**df_ind.to_dict('series'))
df_full = df_full.assign(**df_labels.to_dict('series'))
df_full['crash_quantile'] = quantile_series
df_full['logreturn'] = logret
df_full = df_full.sort_index()

In [13]:
# One block per chart panel. Every column below already exists in df_ind - this is a
# selection, not new indicator math. Three rules govern it:
#   1. At most 3 plotted series per panel. Shaded percentile bands and constant
#      reference lines are references, not series, and do not count.
#   2. No two series in a panel measure the same quantity.
#   3. Every axis is scale-free, rebased at plot time, or bounded - otherwise windows
#      would not be comparable, which is the whole premise of the rolling charts.
#
# Deliberately excluded, and why:
#   williamsr                      -> was stoch_k - 100 exactly; deleted from df_ind
#   stoch_k                        -> ~0.8 correlated with rsi (same 14d range position).
#                                     Panel 7 pairs rsi with adx instead, which measures
#                                     trend strength rather than position.
#   r_5pct                         -> this is the crash label's *unshifted* quantile. The
#                                     label's actual boundary is r_5pct.shift(1), so
#                                     plotting it drew a line that was not the boundary
#                                     and marked breaches the label never fired on.
#   roc_5 / roc_10                 -> strongly collinear with roc_20; one horizon suffices
#   skewness_20 / kurtosis_20      -> deleted; 20-obs higher moments are estimator noise
#   sma20                          -> redundant with the Bollinger midline and with zscore
#   atr, macd*                     -> index points, not comparable between windows
#   obv                            -> cumulative with an arbitrary origin; volume_z_20 is
#                                     the comparable flow measure
#   vol_gk / parkinson / rs        -> alternative estimators of `volatility`, not new info
#   composite_vol / volofvol       -> derived from those estimators
#   cci / sharpe_20 / plus_di /    -> overlap the oscillator, momentum and trend panels
#     minus_di / es_5pct_126
indicators_list_small = [
    # Panel 1 - Trend and regime (rebased to 100 at the window start)
    "close",
    "sma50",
    "sma200",
    "bb_upper",
    "bb_lower",
    "death_cross",
    "golden_cross",

    # Panel 2 - Volatility shock vs regime, inside its own 126d percentile band
    "hv_10",
    "hv_60",
    "volatility",
    "volatility_5pct",
    "volatility_95pct",

    # Panel 3 - Range and squeeze (both scale-free)
    "tr_percent",
    "bb_width",

    # Panel 4 - Distribution shape at the shortest window where it estimates anything
    "skewness_60",
    "kurtosis_60",

    # Panel 5 - Tail risk: the deeper the tail, the longer the window it needs
    "logreturn",
    "var_5pct_60",
    "r_1pct",

    # Panel 6 - Drawdown depth and persistence
    "dd_20",
    "ulcer_index",

    # Panel 7 - Stretch vs trend quality, shared 0-100 axis
    "rsi",
    "adx",

    # Panel 8 - Momentum magnitude and distribution-relative stretch
    "roc_20",
    "zscore",

    # Panel 9 - Flow confirmation
    "cmf",
    "volume_z_20",
]

missing = [col for col in indicators_list_small if col not in df_ind.columns]
assert not missing, f"indicators_list_small references columns df_ind does not have: {missing}"

df_ind_small = df_ind[indicators_list_small]
print(f"Charted indicators: {len(indicators_list_small)} of {len(df_ind.columns)} columns in df_ind")

AssertionError: indicators_list_small references columns df_ind does not have: ['hv_10', 'hv_60', 'volatility', 'volatility_5pct', 'volatility_95pct', 'tr_percent', 'var_5pct_60', 'r_1pct', 'rsi', 'roc_20', 'zscore', 'cmf', 'volume_z_20']

## Output

In [15]:
# 1. OHLCV
df_ohlcv_path = f"{L1_RESULTS}/{L2_DATA}/{OHLCV}.csv"
os.makedirs(os.path.dirname(df_ohlcv_path), exist_ok=True)
df.to_csv(df_ohlcv_path, index=True, header=True)

# 2. Technical Indicators
df_indicators_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}.csv"
os.makedirs(os.path.dirname(df_indicators_path), exist_ok=True)
df_ind.to_csv(df_indicators_path, index=True, header=True)

# 3. Short Technical Indicators
# df_ind_small = df_ind[indicators_list_small]
# df_ind_small_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS}_small.csv"
# os.makedirs(os.path.dirname(df_ind_small_path), exist_ok=True)
# df_ind_small.to_csv(df_ind_small_path, index=True, header=True)

# 4. Crash Labels & Stats (labels, min_future_logrets, thresholds)
df_crash_labels_path = f"{L1_RESULTS}/{L2_DATA}/{CRASH_LABEL}.csv"
os.makedirs(os.path.dirname(df_crash_labels_path), exist_ok=True)
df_labels = pd.DataFrame({
    "date": price_df.index,
    "crash_label": labels,
    "min_future_logret": min_future_logrets,
    "crash_threshold": thresholds
}).set_index("date")
df_labels.to_csv(df_crash_labels_path, index=True, header=True)

# 5. Rolling crash quantile/thresh series
df_quantile_series_path = f"{L1_RESULTS}/{L2_DATA}/{QUANTILE}.csv"
os.makedirs(os.path.dirname(df_quantile_series_path), exist_ok=True)
quantile_series.to_frame("crash_quantile").to_csv(df_quantile_series_path, index=True, header=True)

# 6. Log returns
df_logret_path = f"{L1_RESULTS}/{L2_DATA}/{LOG_RETURNS}.csv"
os.makedirs(os.path.dirname(df_logret_path), exist_ok=True)
logret.to_frame("logreturn").to_csv(df_logret_path, index=True, header=True)

# 7. Valid dates
df_valid_dates_path = f"{L1_RESULTS}/{L2_DATA}/{VALID_DATES}.csv"
os.makedirs(os.path.dirname(df_valid_dates_path), exist_ok=True)
pd.DataFrame({"date": valid_dates}).to_csv(df_valid_dates_path, index=False, header=True)

# 8. Full DataFrame
df_full_path = f"{L1_RESULTS}/{L2_DATA}/{FULL}.csv"
os.makedirs(os.path.dirname(df_full_path), exist_ok=True)
df_full.to_csv(df_full_path, index=True, header=True)

# 9. Full Indicators List
indicators_list_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS_LIST}.txt"
with open(indicators_list_path, "w") as f:
    for col in df_ind.columns:
        f.write(f"{col}\n")

# 10. Short Indicators List
indicators_list_small_path = f"{L1_RESULTS}/{L2_DATA}/{INDICATORS_LIST}_small.txt"
os.makedirs(os.path.dirname(indicators_list_small_path), exist_ok=True)
with open(indicators_list_small_path, "w") as f:
    for col in indicators_list_small:
        f.write(f"{col}\n")

In [49]:
# NEWS_CACHE_PATH = f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv"

# print(f"Fetching news for {NEWS_QUERY} ...")
# news_df = fetch_and_cache_news(
#     NEWS_QUERY, START_DATE, END_DATE, max_results=5, chunk_size=3, cache_path=NEWS_CACHE_PATH
# )
# print("News data loaded:", news_df.shape)

In [50]:
# NEWS_CACHE_PATH = f"{NEWS_CACHE}.csv"
# news_df = pd.read_csv(NEWS_CACHE_PATH, index_col='Date', parse_dates=True)
# shutil.copy(NEWS_CACHE_PATH, f"{L1_RESULTS}/{L2_DATA}/{NEWS_CACHE}.csv")

# df_full = df_full.assign(**news_df.to_dict('series'))
# df_full = df_full.sort_index()
# df_full.to_csv(df_full_path, index=True, header=True)